<a href="https://colab.research.google.com/github/dks1532/SNU_BigData_AI_Fintech/blob/Web_Development/%EC%9B%B9%EA%B0%9C%EB%B0%9C_%EB%B0%8F_%EC%8B%9C%EA%B0%81%ED%99%94_1%EC%9D%BC%EC%B0%A8(260723).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd

# Melon 실시간 차트 URL
url = 'https://www.melon.com/chart/index.htm'
headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/119.0.0.0 Safari/537.36'
}

response = requests.get(url, headers=headers)
soup = BeautifulSoup(response.text, 'html.parser')

# 데이터 추출
songs = soup.select('tr.lst50, tr.lst100')

chart_data = []
for song in songs:
    rank = song.select_one('.rank').text
    title = song.select_one('.rank01 span a').text
    artist = song.select_one('.rank02 span a').text
    album = song.select_one('.rank03 a').text

    chart_data.append({
        '순위': rank,
        '곡제목': title,
        '아티스트': artist,
        '앨범': album
    })

# 데이터프레임 생성 및 CSV 저장
df = pd.DataFrame(chart_data)
df.to_csv('melon_chart_top100.csv', index=False, encoding='utf-8-sig')

print('데이터 추출 및 melon_chart_top100.csv 저장 완료')
display(df.head())

데이터 추출 및 melon_chart_top100.csv 저장 완료


,순위,곡제목,아티스트,앨범
0,1,LOVE ATTACK,RESCENE (리센느),SCENEDROME
1,2,갑자기,아이오아이 (I.O.I),I.O.I 3rd MINI ALBUM [I.O.I : LOOP]
2,3,REDRED,CORTIS (코르티스),GREENGREEN
3,4,LEMONADE,aespa,LEMONADE - The 2nd Album
4,5,It′s Me,아일릿(ILLIT),MAMIHLAPINATAPAI


In [8]:
import requests
import pandas as pd
import math

# API 엔드포인트 및 기본 파라미터
api_base_url = 'https://www.yestrade.go.kr/api/hsk-control-search'

# 초기 요청을 통해 총 데이터 개수 확인
initial_params = {
    'page': 1,
    'searchType': 'hskCd',
    'searchKeyword': '',
    'recordPerPage': 10 # API 응답의 'pageSize'가 10이므로 여기에 맞춰서 요청해야 정확한 totalCount를 얻을 수 있음
}

# 사용자 에이전트 헤더 추가
headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/119.0.0.0 Safari/537.36'
}

all_data = []

try:
    # 첫 페이지 요청
    print("초기 데이터 요청 중...")
    response = requests.get(api_base_url, params=initial_params, headers=headers)
    response.raise_for_status() # HTTP 오류가 발생하면 예외 발생
    json_data = response.json()

    # total_count와 record_per_page를 pageData 객체에서 추출
    page_data = json_data.get('pageData', {})
    total_count = page_data.get('totalCount', 0)
    record_per_page = page_data.get('pageSize', 10) # API 응답에서 페이지당 레코드 수 가져오기 (기본값 10)

    if total_count == 0:
        print("API에서 총 데이터 개수를 찾을 수 없습니다. 또는 데이터가 없습니다.")
    else:
        max_items_to_fetch = 100
        # 1000개 데이터를 가져오기 위해 필요한 페이지 수 계산
        limited_pages = math.ceil(max_items_to_fetch / record_per_page)
        total_pages = min(math.ceil(total_count / record_per_page), limited_pages)

        print(f"총 {total_count}개의 데이터 중 약 {max_items_to_fetch}개(총 {total_pages} 페이지)를 가져올 예정입니다.")

        for page_num in range(1, total_pages + 1):
            params = {
                'page': page_num,
                'searchType': 'hskCd',
                'searchKeyword': '',
                'recordPerPage': record_per_page
            }

            print(f"페이지 {page_num}/{total_pages} 데이터 요청 중...")
            page_response = requests.get(api_base_url, params=params, headers=headers)
            page_response.raise_for_status()
            page_json = page_response.json()

            # 실제 데이터는 pageData 내부의 items 리스트에 있음
            current_page_data = page_json.get('pageData', {})
            items = current_page_data.get('items', [])

            if items:
                all_data.extend(items)
                if len(all_data) >= max_items_to_fetch:
                    print(f"요청한 {max_items_to_fetch}개 이상의 데이터를 수집하여 중지합니다.")
                    break
            else:
                print(f"페이지 {page_num}에 데이터가 없습니다. 중지합니다.")
                break

        if all_data:
            df_yestrade = pd.DataFrame(all_data)

            # 컬럼명 매핑 (API 필드명 -> 한글명)
            column_map = {
                'hskCd': '품목분류(HS)',
                'hskNm': 'HSK품목명',
                'hskEngNm': 'HSK영문명',
                'ctlNo': '통제번호'
            }
            df_yestrade = df_yestrade.rename(columns=column_map)

            df_yestrade.to_csv('yestrade_dual_use_data.csv', index=False, encoding='utf-8-sig')
            print(f"총 {len(df_yestrade)}개의 데이터를 성공적으로 추출하여 'yestrade_dual_use_data.csv'로 저장했습니다.")
            display(df_yestrade.head(10))
        else:
            print('수집된 데이터가 없습니다.')

except requests.exceptions.RequestException as e:
    print(f"API 요청 중 오류 발생: {e}")
except Exception as e:
    print(f"예상치 못한 오류 발생: {e}")

초기 데이터 요청 중...
총 20393개의 데이터 중 약 100개(총 10 페이지)를 가져올 예정입니다.
페이지 1/10 데이터 요청 중...
페이지 2/10 데이터 요청 중...
페이지 3/10 데이터 요청 중...
페이지 4/10 데이터 요청 중...
페이지 5/10 데이터 요청 중...
페이지 6/10 데이터 요청 중...
페이지 7/10 데이터 요청 중...
페이지 8/10 데이터 요청 중...
페이지 9/10 데이터 요청 중...
페이지 10/10 데이터 요청 중...
요청한 100개 이상의 데이터를 수집하여 중지합니다.
총 100개의 데이터를 성공적으로 추출하여 'yestrade_dual_use_data.csv'로 저장했습니다.


,품목분류(HS),HSK영문명,HSK품목명,cntrlNo,cntrlNoLvl,upCntrlNo,cntrlSttsCd,sortCntrlNo
0,3921199090,Other,기타,1A001.,1,1A.,04,1A001.
1,3920999020,Of fluorine polyimide,불화폴리이미드로 만든 것,1A001.,1,1A.,04,1A001.
2,3920991000,For aircrafts,항공기용,1A001.,1,1A.,04,1A001.
3,9620000000,"Monopods, bipods, tripods and similar articles.",일각대ㆍ양각대ㆍ삼각대와 이와 유사한 물품,1A001.,1,1A.,04,1A001.
4,3920999090,Other,기타,1A001.,1,1A.,04,1A001.
5,3920999010,"Polyimides film, for manufacturing Printed Cir...",폴리이미드 필름(리드프레임의 기능을 하는 인쇄회로기판 제조용으로 한정한다),1A001.,1,1A.,04,1A001.
6,3921909090,Other,기타,1A001.,1,1A.,04,1A001.
7,8484900000,Other,기타,1A001.,1,1A.,04,1A001.
8,8807100000,Propellers and rotors and parts thereof,프로펠러ㆍ로터(rotor)와 이들의 부분품,1A001.,1,1A.,04,1A001.
9,3917400000,Fittings,연결구류,1A001.a.,2,1A001.,04,1A001.a.
